# Cross-MLLM Confusion Matrix Analysis

Computes confusion matrices and inter-rater agreement statistics from the two JSONL result files (`gemini_results.jsonl`, `qwen_results.jsonl`) produced by the analysis notebooks. Outputs:

- **Per-generator confusion matrix** for each MLLM (Tables 1 and 2 for the Results chapter)
- **Binary classification metrics** — accuracy, precision, recall, F1, FPR, FNR (Table 3)
- **Inter-MLLM cross-tabulation** with Cohen's kappa and Krippendorff's alpha (Table 4, RQ2)
- **Per-generator difficulty table** — which generators each MLLM misses (Table 5)
- **Heatmap figures** saved to Drive as PNG for insertion into the dissertation
- **Everything saved to CSV** so you can regenerate tables without re-running

## Before you run
1. Mount Drive — both JSONL files must exist at the paths in Cell 3.
2. Run cells in order. Total runtime ~30 seconds.


## 1. Install and import

In [ ]:
!pip install -q matplotlib numpy
import json, pathlib, csv, math
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')


## 2. Configure paths

In [ ]:
# INPUT — the two JSONL logs produced by the analysis notebooks
GEMINI_JSONL = "/content/drive/MyDrive/msc-deepfake/gemini_full_run/gemini_results.jsonl"
QWEN_JSONL   = "/content/drive/MyDrive/msc-deepfake/qwen_full_run/qwen_results.jsonl"

# OUTPUT — everything (CSVs, PNGs) goes here
OUT_DIR = pathlib.Path("/content/drive/MyDrive/msc-deepfake/analysis_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Reading from:\n  {GEMINI_JSONL}\n  {QWEN_JSONL}")
print(f"Writing to:\n  {OUT_DIR}")

# Canonical ordering used everywhere
GENERATORS = ["LTX", "Hunyuan", "Wan", "Kling", "Gemini_Omni", "Seedance", "Pexels"]
EXPECTED   = {"LTX": 40, "Hunyuan": 40, "Wan": 40,
              "Kling": 32, "Gemini_Omni": 32, "Seedance": 32, "Pexels": 64}
VERDICTS   = ["AI-generated", "Real", "Uncertain"]


## 3. Load and deduplicate

The JSONL files may contain multiple entries for a video if the analysis was re-run. This cell keeps the most recent successful record per `video_id`.


In [ ]:
def load_jsonl(path):
    recs = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                recs.append(json.loads(line))
            except json.JSONDecodeError:
                pass   # malformed line, skip

    # Dedupe: prefer 'ok' status, then latest timestamp
    by_id = {}
    for r in recs:
        vid = r.get("video_id")
        if not vid:
            continue
        cur = by_id.get(vid)
        if cur is None:
            by_id[vid] = r
            continue
        cur_ok = cur.get("status") == "ok"
        new_ok = r.get("status") == "ok"
        if new_ok and not cur_ok:
            by_id[vid] = r
        elif new_ok == cur_ok and r.get("timestamp", "") > cur.get("timestamp", ""):
            by_id[vid] = r
    return list(by_id.values())

gem  = load_jsonl(GEMINI_JSONL)
qwen = load_jsonl(QWEN_JSONL)
print(f"Gemini unique videos: {len(gem)}")
print(f"Qwen unique videos:   {len(qwen)}")
print(f"Gemini status: {dict(Counter(r.get('status') for r in gem))}")
print(f"Qwen status:   {dict(Counter(r.get('status') for r in qwen))}")


## 4. Per-generator verdict matrices

For each MLLM, build a matrix with rows = generator, columns = verdict (AI-generated / Real / Uncertain). This directly answers RQ3's per-generator breakdown.


In [ ]:
def build_matrix(records):
    m = {g: {"AI-generated": 0, "Real": 0, "Uncertain": 0, "error": 0} for g in GENERATORS}
    for r in records:
        gen = r.get("generator")
        if gen not in m:
            continue
        if r.get("status") != "ok":
            m[gen]["error"] += 1
            continue
        v = (r.get("parsed") or {}).get("video_verdict", "?")
        if v in m[gen]:
            m[gen][v] += 1
        else:
            m[gen]["error"] += 1
    return m

gem_matrix  = build_matrix(gem)
qwen_matrix = build_matrix(qwen)

def print_matrix(name, m):
    print(f"\n=== {name} ===")
    print(f"{'Generator':14s} {'AI-gen':>8s} {'Real':>6s} {'Uncert':>7s} {'Err':>5s}  {'n':>4s}")
    for g in GENERATORS:
        row = m[g]
        n = row["AI-generated"] + row["Real"] + row["Uncertain"] + row["error"]
        print(f"{g:14s} {row['AI-generated']:8d} {row['Real']:6d} {row['Uncertain']:7d} {row['error']:5d}  {n:4d}")

print_matrix("Gemini 3.1 Pro", gem_matrix)
print_matrix("Qwen 3.5-397B-A17B", qwen_matrix)


## 5. Binary classification metrics

Collapse the per-generator matrices into a single binary decision (AI-generated vs Real) and compute the standard classification metrics justified in the methodology chapter: accuracy, precision, recall, F1, false-positive rate, false-negative rate.

Ground truth: Pexels is Real; every other source is AI-generated. "Uncertain" verdicts are treated as non-detections (not counted as AI-generated predictions), giving false negatives on the AI class and no effect on the Real class.


In [ ]:
def binary_stats(m):
    tp = fp = tn = fn = 0
    for g in GENERATORS:
        row = m[g]
        if g == "Pexels":
            tn += row["Real"]           # correct — real called real
            fp += row["AI-generated"]   # wrong  — real called AI
            # 'Uncertain' on Pexels: not counted as AI, so contributes to neither TP nor FP.
            # Conservatively counted here as no-decision (excluded from FPR denominator? no —
            # standard practice: keep in the "negative" pool. Since it's not FP, it's TN-like
            # in that no false alarm was raised.)
            tn += row["Uncertain"]
        else:
            tp += row["AI-generated"]   # correct — AI called AI
            fn += row["Real"]           # wrong  — AI called real
            fn += row["Uncertain"]      # AI, no positive decision = false negative
    return dict(TP=tp, FP=fp, TN=tn, FN=fn)

def metrics(s):
    tp, fp, tn, fn = s["TP"], s["FP"], s["TN"], s["FN"]
    total = tp + fp + tn + fn
    acc  = (tp + tn) / total if total else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    fpr  = fp / (fp + tn) if (fp + tn) else 0.0
    fnr  = fn / (fn + tp) if (fn + tp) else 0.0
    return dict(accuracy=acc, precision=prec, recall=rec, f1=f1, fpr=fpr, fnr=fnr)

gem_stats,  gem_met  = binary_stats(gem_matrix),  None
qwen_stats, qwen_met = binary_stats(qwen_matrix), None
gem_met  = metrics(gem_stats)
qwen_met = metrics(qwen_stats)

print(f"{'':16s} {'TP':>5s} {'FP':>5s} {'TN':>5s} {'FN':>5s}  {'Acc':>6s} {'Prec':>6s} {'Rec':>6s} {'F1':>6s}  {'FPR':>6s} {'FNR':>6s}")
for name, s, mt in [("Gemini 3.1 Pro", gem_stats, gem_met),
                     ("Qwen 3.5-397B",  qwen_stats, qwen_met)]:
    print(f"{name:16s} {s['TP']:5d} {s['FP']:5d} {s['TN']:5d} {s['FN']:5d}  "
          f"{mt['accuracy']:6.3f} {mt['precision']:6.3f} {mt['recall']:6.3f} {mt['f1']:6.3f}  "
          f"{mt['fpr']:6.3f} {mt['fnr']:6.3f}")


## 6. Inter-MLLM agreement (RQ2)

Compute Cohen's kappa and Krippendorff's alpha on the verdict for the subset of videos where both MLLMs succeeded. Alpha uses nominal-level agreement here since the three verdict categories are unordered.


In [ ]:
gem_by_id  = {r["video_id"]: r for r in gem  if r.get("status") == "ok"}
qwen_by_id = {r["video_id"]: r for r in qwen if r.get("status") == "ok"}
shared = sorted(set(gem_by_id) & set(qwen_by_id))
print(f"Shared videos (both MLLMs OK): {len(shared)}")

def verdict(r):
    return (r.get("parsed") or {}).get("video_verdict", "?")

pairs = [(verdict(gem_by_id[v]), verdict(qwen_by_id[v])) for v in shared]

# Cross-tabulation
cross = {(a, b): 0 for a in VERDICTS for b in VERDICTS}
for a, b in pairs:
    if a in VERDICTS and b in VERDICTS:
        cross[(a, b)] += 1

print("\nGemini (rows) \u00d7 Qwen (columns):")
print(f"{'':14s}" + "".join(f"{c:>13s}" for c in VERDICTS))
for a in VERDICTS:
    print(f"{a:14s}" + "".join(f"{cross[(a,b)]:13d}" for b in VERDICTS))

# Cohen's kappa
n = len(pairs)
agree = sum(cross[(c, c)] for c in VERDICTS)
po = agree / n if n else 0.0
gem_marg  = Counter(a for a, _ in pairs)
qwen_marg = Counter(b for _, b in pairs)
pe = sum((gem_marg[c]/n) * (qwen_marg[c]/n) for c in VERDICTS) if n else 0.0
kappa = (po - pe) / (1 - pe) if pe < 1 else 0.0

# Krippendorff's alpha for two coders, nominal data
# alpha = 1 - Do/De, where Do = observed disagreement, De = expected disagreement.
# For 2 coders on a single value each per unit, Do and De computed on the coincidence matrix.
# Build the coincidence matrix.
coinc = {(a, b): 0.0 for a in VERDICTS for b in VERDICTS}
for a, b in pairs:
    coinc[(a, b)] += 1.0
    coinc[(b, a)] += 1.0
# Row sums = column sums = 2 * marginal counts (each pair contributes to both)
row_sums = {c: sum(coinc[(c, x)] for x in VERDICTS) for c in VERDICTS}
n_total  = sum(row_sums.values())
# Observed disagreement (off-diagonal contributions, nominal metric = 0 if equal, 1 else)
Do_num = sum(coinc[(a, b)] for a in VERDICTS for b in VERDICTS if a != b)
Do = Do_num / n_total if n_total else 0.0
# Expected disagreement
De_num = sum(row_sums[a] * row_sums[b] for a in VERDICTS for b in VERDICTS if a != b)
De_denom = n_total * (n_total - 1) if n_total > 1 else 1
De = De_num / De_denom
alpha = 1 - Do / De if De else 0.0

print(f"\nRaw agreement:      {po:.3f}  ({agree}/{n})")
print(f"Expected by chance: {pe:.3f}")
print(f"Cohen's kappa:      {kappa:.3f}")
print(f"Krippendorff alpha: {alpha:.3f} (nominal, 2 coders)")

# Interpretation by Landis & Koch (1977) bands
def interpret(k):
    if k < 0.00: return "poor / worse than chance"
    if k < 0.21: return "slight"
    if k < 0.41: return "fair"
    if k < 0.61: return "moderate"
    if k < 0.81: return "substantial"
    return "almost perfect"
print(f"Kappa interpretation (Landis & Koch, 1977): {interpret(kappa)}")


## 7. Per-generator difficulty (which generators are hardest to detect)

For each generator, count the videos each MLLM misses (predicts Real or Uncertain when the ground truth is AI). This directly answers Prof. Li's question about which generators produce videos hardest to detect.


In [ ]:
ai_generators = [g for g in GENERATORS if g != "Pexels"]
print(f"{'Generator':14s} {'Missed by Gemini':>20s} {'Missed by Qwen':>18s}")
difficulty = []
for g in ai_generators:
    g_rows = gem_matrix[g]
    q_rows = qwen_matrix[g]
    g_miss = g_rows["Real"] + g_rows["Uncertain"]
    q_miss = q_rows["Real"] + q_rows["Uncertain"]
    g_n = g_miss + g_rows["AI-generated"]
    q_n = q_miss + q_rows["AI-generated"]
    difficulty.append((g, g_miss, g_n, q_miss, q_n))
    print(f"{g:14s} {g_miss:3d} / {g_n:2d} ({g_miss/g_n*100:5.1f}%)  {q_miss:3d} / {q_n:2d} ({q_miss/q_n*100:5.1f}%)")


## 8. Save all tables as CSV

Everything computed above is written to CSV in the analysis outputs folder, so the tables in the dissertation can be regenerated without re-running the whole analysis.


In [ ]:
# Table 1 & 2: per-generator verdicts
for name, mat in [("gemini", gem_matrix), ("qwen", qwen_matrix)]:
    path = OUT_DIR / f"table_verdict_by_generator_{name}.csv"
    with path.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["generator", "n_expected", "AI_generated", "Real", "Uncertain", "error"])
        for g in GENERATORS:
            row = mat[g]
            w.writerow([g, EXPECTED[g], row["AI-generated"], row["Real"], row["Uncertain"], row["error"]])
    print(f"Wrote {path}")

# Table 3: binary metrics
path = OUT_DIR / "table_binary_metrics.csv"
with path.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["metric", "gemini", "qwen"])
    w.writerow(["TP", gem_stats["TP"], qwen_stats["TP"]])
    w.writerow(["FP", gem_stats["FP"], qwen_stats["FP"]])
    w.writerow(["TN", gem_stats["TN"], qwen_stats["TN"]])
    w.writerow(["FN", gem_stats["FN"], qwen_stats["FN"]])
    for k in ["accuracy", "precision", "recall", "f1", "fpr", "fnr"]:
        w.writerow([k, f"{gem_met[k]:.4f}", f"{qwen_met[k]:.4f}"])
print(f"Wrote {path}")

# Table 4: cross-MLLM
path = OUT_DIR / "table_cross_mllm.csv"
with path.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["gemini_verdict", "qwen_verdict", "count"])
    for a in VERDICTS:
        for b in VERDICTS:
            w.writerow([a, b, cross[(a, b)]])
    w.writerow([])
    w.writerow(["n", n])
    w.writerow(["cohen_kappa", f"{kappa:.4f}"])
    w.writerow(["krippendorff_alpha_nominal", f"{alpha:.4f}"])
    w.writerow(["observed_agreement", f"{po:.4f}"])
    w.writerow(["expected_agreement", f"{pe:.4f}"])
print(f"Wrote {path}")

# Table 5: difficulty per generator
path = OUT_DIR / "table_difficulty_per_generator.csv"
with path.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["generator", "missed_by_gemini", "n_gemini", "missed_by_qwen", "n_qwen"])
    for g, g_miss, g_n, q_miss, q_n in difficulty:
        w.writerow([g, g_miss, g_n, q_miss, q_n])
print(f"Wrote {path}")


## 9. Figures — heatmaps

Three PNG heatmaps at 150 dpi, suitable for inclusion in the Results chapter.


In [ ]:
LABELS_GEN = ["LTX", "Hunyuan", "Wan", "Kling", "Gemini\nOmni", "Seedance", "Pexels\n(Real)"]

def plot_matrix(matrix, title, filename, cmap='Blues'):
    data = np.array([[matrix[g][v] for v in VERDICTS] for g in GENERATORS])
    row_totals = data.sum(axis=1, keepdims=True); row_totals[row_totals == 0] = 1
    normed = data / row_totals

    fig, ax = plt.subplots(figsize=(7.2, 5.4))
    ax.imshow(normed, cmap=cmap, aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(VERDICTS)));  ax.set_xticklabels(VERDICTS, fontsize=11)
    ax.set_yticks(range(len(GENERATORS))); ax.set_yticklabels(LABELS_GEN, fontsize=10)
    for i, lbl in enumerate(ax.get_yticklabels()):
        if GENERATORS[i] == "Pexels":
            lbl.set_color("#8b0000"); lbl.set_fontweight('bold')
    for i in range(len(GENERATORS)):
        for j in range(len(VERDICTS)):
            v = data[i, j]; pct = normed[i, j] * 100
            colour = 'white' if normed[i, j] > 0.55 else '#222'
            ax.text(j, i, f"{v}\n({pct:.0f}%)", ha='center', va='center',
                    color=colour, fontsize=10, fontweight='bold')
    ax.set_xlabel("Predicted verdict", fontsize=11, fontweight='bold')
    ax.set_ylabel("Ground truth (source)", fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=12)
    ax.set_xticks(np.arange(-.5, len(VERDICTS), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(GENERATORS), 1), minor=True)
    ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5, alpha=0.4)
    ax.tick_params(which='minor', bottom=False, left=False)
    plt.tight_layout(); plt.savefig(filename, dpi=150, bbox_inches='tight'); plt.close()
    print(f"Wrote {filename}")

plot_matrix(gem_matrix,  "Gemini 3.1 Pro \u2014 verdict per generator",
            OUT_DIR / "fig_cm_gemini.png", cmap='Blues')
plot_matrix(qwen_matrix, "Qwen 3.5-397B-A17B \u2014 verdict per generator",
            OUT_DIR / "fig_cm_qwen.png",   cmap='Oranges')

# Cross-MLLM heatmap
data = np.array([[cross[(a, b)] for b in VERDICTS] for a in VERDICTS])
fig, ax = plt.subplots(figsize=(5.6, 4.2))
ax.imshow(data, cmap='Purples', aspect='auto')
ax.set_xticks(range(len(VERDICTS))); ax.set_xticklabels(VERDICTS, fontsize=10)
ax.set_yticks(range(len(VERDICTS))); ax.set_yticklabels(VERDICTS, fontsize=10)
for i in range(len(VERDICTS)):
    for j in range(len(VERDICTS)):
        v = data[i, j]
        colour = 'white' if v > data.max()*0.6 else '#222'
        ax.text(j, i, f"{v}", ha='center', va='center',
                color=colour, fontsize=13, fontweight='bold')
ax.set_xlabel("Qwen 3.5 verdict", fontsize=11, fontweight='bold')
ax.set_ylabel("Gemini 3.1 Pro verdict", fontsize=11, fontweight='bold')
ax.set_title(f"Inter-MLLM cross-tab (n={data.sum()}) \u2014 "
             f"Cohen \u03BA = {kappa:.3f}, Krippendorff \u03B1 = {alpha:.3f}",
             fontsize=11, fontweight='bold', pad=10)
ax.set_xticks(np.arange(-.5, len(VERDICTS), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(VERDICTS), 1), minor=True)
ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5, alpha=0.4)
ax.tick_params(which='minor', bottom=False, left=False)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_cross_mllm.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"Wrote {OUT_DIR / 'fig_cross_mllm.png'}")

print("\nAll files saved. Insert the PNGs into the Results chapter and cite the CSVs as data appendices.")


## 10. Cell for the dissertation appendix

If you need to describe the computation in the Methodology or Appendix, this cell prints a concise summary of what was done. Copy the printed lines into your text.


In [ ]:
print("Method summary — for Methodology chapter or appendix caption")
print("-" * 70)
print(f"Both MLLMs analysed the same 280-video corpus with the same Variant B")
print(f"taxonomy-guided prompt at temperature 0.")
print(f"")
print(f"Ground truth: source label. Pexels = Real; all other sources = AI-generated.")
print(f"")
print(f"Per-generator matrices report the raw verdict distribution for each source,")
print(f"row-normalised for figures.")
print(f"")
print(f"Binary metrics collapse the six AI generators into a single positive class.")
print(f"'Uncertain' verdicts count as non-detections (contribute to false negatives on")
print(f"AI content; contribute to true negatives on Pexels since no false alarm was raised).")
print(f"")
print(f"Inter-rater statistics computed on the {n} videos where both MLLMs succeeded.")
print(f"Cohen's kappa follows Cohen (1960); Krippendorff's alpha uses the nominal metric")
print(f"following Krippendorff (2004), computed on the coincidence matrix.")
print(f"")
print(f"Interpretation bands for kappa follow Landis and Koch (1977).")
